In [ ]:
# =============================================================================
# Random Forest Wildfire Ignition + Cause Model — Ontario
# Train: 2010–2020  |  Validation: 2021–2022
# Ignition: RF with 20:1 undersampling, threshold = 0.10
# Cause: two binary RF models (Human vs Other, Natural vs Other)
#        combined via three-way decision rule
# =============================================================================

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, roc_auc_score,
                             confusion_matrix, balanced_accuracy_score,
                             precision_score, recall_score, f1_score,
                             average_precision_score)
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import glob
import os
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ─────────────────────────────────────────────────────────────
DATA_DIR = r'C:\Users\lambe\RU_Thesis_2026\ml_data'
os.makedirs('models',  exist_ok=True)
os.makedirs('figures', exist_ok=True)

DYNAMIC_FEATURES = [
    'temp_max', 'rh_min', 'vpd_mean', 'wind_speed_mean',
    'precip_sum', 'solar_rad_mj', 'rh_7d', 'temp_7d',
    'precip_30d', 'recovery_value'
]
STATIC_FEATURES = [
    'elevation_m', 'slope_deg', 'land_cover',
    'pop_density', 'road_proximity_m'
]
FEATURES = DYNAMIC_FEATURES + STATIC_FEATURES

FEATURE_LABELS = {
    'temp_max'        : 'Daily max temperature (°C)',
    'rh_min'          : 'Daily min relative humidity (%)',
    'vpd_mean'        : 'Mean VPD (kPa)',
    'wind_speed_mean' : 'Mean wind speed (m/s)',
    'precip_sum'      : 'Daily precipitation (mm)',
    'solar_rad_mj'    : 'Solar radiation (MJ/m²)',
    'rh_7d'           : '7-day mean RH (%)',
    'temp_7d'         : '7-day mean temperature (°C)',
    'precip_30d'      : '30-day precipitation (mm)',
    'recovery_value'  : 'Post-fire recovery [0–1]',
    'elevation_m'     : 'Elevation (m)',
    'slope_deg'       : 'Slope (degrees)',
    'land_cover'      : 'Land cover class',
    'pop_density'     : 'Population density',
    'road_proximity_m': 'Distance to road (m)',
}

DOY_START_HARD      = 91
DOY_END_HARD        = 310
TRAIN_YEARS         = list(range(2010, 2021))
VAL_YEARS           = [2021, 2022]
TARGET_RATIO        = 20
IGNITION_THRESHOLD  = 0.10
UNCERTAIN_THRESHOLD = 0.35
RANDOM_STATE        = 42

COL_DYNAMIC  = '#27ae60'
COL_STATIC   = '#e67e22'
COL_HUMAN    = '#e74c3c'
COL_NATURAL  = '#2980b9'
COL_UNCERT   = '#95a5a6'
COL_GRID     = '#eeeeee'
FONT_TITLE   = 12
FONT_LABEL   = 10
FONT_TICK    = 9

SOURCE_IGN   = ('Ontario wildfire ignition model  |  Random Forest  |  '
                'Training: 2010–2020  |  Validation: 2021–2022')
SOURCE_CAUSE = ('Ontario wildfire cause model  |  Random Forest Binary  |  '
                'Training: 2010–2020  |  Validation: 2021–2022')

def feature_colour(feat):
    return COL_DYNAMIC if feat in DYNAMIC_FEATURES else COL_STATIC

# =============================================================================
# STEP 1: Load and prepare data
# =============================================================================
print("="*60)
print("STEP 1: Loading data")
print("="*60)

def load_years(years, split_label):
    all_dfs = []
    for year in years:
        pattern  = os.path.join(DATA_DIR, f'ML_{split_label}_{year}??.csv')
        files    = sorted(glob.glob(pattern))
        year_dfs = []
        for f in files:
            try:
                year_dfs.append(pd.read_csv(f))
            except Exception as e:
                print(f"  WARNING: {os.path.basename(f)}: {e}")
        if year_dfs:
            df_year = pd.concat(year_dfs, ignore_index=True)
            n_fire  = (df_year['target_ignition'] == 1).sum()
            print(f"  {year}: {len(df_year):>10,} rows  ({n_fire:,} fires)")
            all_dfs.append(df_year)
    return pd.concat(all_dfs, ignore_index=True)

def clean_and_filter(df, name):
    required = FEATURES + ['target_ignition', 'fire_cause', 'date']
    df = df.dropna(subset=required)
    df['target_ignition'] = df['target_ignition'].astype(int)
    df['fire_cause']      = df['fire_cause'].astype(int)
    df['land_cover']      = df['land_cover'].astype(int)
    df = df[df['target_ignition'].isin([0, 1])]
    df = df[df['fire_cause'].between(0, 5)]
    # Date parsing — stored as float
    df['date_str'] = df['date'].astype(int).astype(str)
    df['doy']      = pd.to_datetime(
        df['date_str'], format='%Y%m%d', errors='coerce').dt.dayofyear
    n_before = len(df)
    df = df[(df['doy'] >= DOY_START_HARD) &
            (df['doy'] <= DOY_END_HARD)].copy()
    df = df.drop(columns=['date_str', 'doy'])
    df = df.sort_values('date').reset_index(drop=True)
    print(f"  {name}: {n_before:,} → {len(df):,} rows after season filter")
    return df

print("\nLoading training data (2010–2020)...")
df_train = clean_and_filter(load_years(TRAIN_YEARS, 'train'), 'train')

print("\nLoading validation data (2021–2022)...")
df_val = clean_and_filter(load_years(VAL_YEARS, 'val'), 'val')

X_train     = df_train[FEATURES].values
y_ign_train = df_train['target_ignition'].values
X_val       = df_val[FEATURES].values
y_ign_val   = df_val['target_ignition'].values

fire_tr_mask = df_train['target_ignition'] == 1
fire_vl_mask = df_val['target_ignition']   == 1

print(f"\n  Train: {X_train.shape}  fires: {fire_tr_mask.sum():,}")
print(f"  Val  : {X_val.shape}    fires: {fire_vl_mask.sum():,}")

# =============================================================================
# STEP 2: Train RF Ignition Model
# =============================================================================
print("\n" + "="*60)
print("STEP 2: Training RF Ignition Model")
print(f"  Threshold: {IGNITION_THRESHOLD}  |  Undersampling: {TARGET_RATIO}:1")
print("="*60)

n_fire_tr    = fire_tr_mask.sum()
n_nofire_smp = int(n_fire_tr * TARGET_RATIO)
nofire_idx   = np.where(~fire_tr_mask.values)[0]
sampled_idx  = np.random.RandomState(RANDOM_STATE).choice(
    nofire_idx, size=n_nofire_smp, replace=False)
combined_idx = np.concatenate(
    [np.where(fire_tr_mask.values)[0], sampled_idx])
np.random.RandomState(RANDOM_STATE).shuffle(combined_idx)

X_train_bal = X_train[combined_idx]
y_train_bal = y_ign_train[combined_idx]

print(f"\n  Balanced train: {len(X_train_bal):,}  "
      f"({n_fire_tr:,} fire + {n_nofire_smp:,} no-fire)")

t0 = time.time()
model_rf_ign = RandomForestClassifier(
    n_estimators     = 300,
    max_depth        = None,
    max_features     = 'sqrt',
    min_samples_leaf = 5,
    class_weight     = {0: 1, 1: TARGET_RATIO},
    oob_score        = True,
    n_jobs           = -1,
    random_state     = RANDOM_STATE,
    verbose          = 1
)
model_rf_ign.fit(X_train_bal, y_train_bal)
print(f"\n  Training time: {(time.time()-t0)/60:.1f} min  "
      f"OOB: {model_rf_ign.oob_score_:.4f}")

# ── Ignition evaluation ────────────────────────────────────────────────────────
y_prob_ign = model_rf_ign.predict_proba(X_val)[:, 1]
y_pred_ign = (y_prob_ign >= IGNITION_THRESHOLD).astype(int)

tn, fp, fn, tp = confusion_matrix(y_ign_val, y_pred_ign).ravel()
print(f"\n  Ignition validation (threshold={IGNITION_THRESHOLD}):")
print(f"    ROC-AUC  : {roc_auc_score(y_ign_val, y_prob_ign):.4f}")
print(f"    Recall   : {tp/(tp+fn):.4f}  ({tp} of {tp+fn} fires)")
print(f"    Precision: {tp/(tp+fp) if (tp+fp)>0 else 0:.4f}")
print(f"    FPR      : {fp/(fp+tn):.4f}")
print(f"    TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}")
print(classification_report(y_ign_val, y_pred_ign,
                            target_names=['No fire', 'Fire'],
                            digits=4, zero_division=0))

joblib.dump(model_rf_ign,
            'models/model_rf_ignition_2010_2022_ratio20.pkl')
print("  Saved: models/model_rf_ignition_2010_2022_ratio20.pkl")

# =============================================================================
# STEP 3: Identify predicted fire pixels for cause modelling
# =============================================================================
print("\n" + "="*60)
print("STEP 3: Identifying predicted fire pixels for cause modelling")
print("="*60)

# Training: use ACTUAL fire pixels (confirmed ignitions with known cause)
X_cause_train   = df_train.loc[fire_tr_mask, FEATURES].values
y_cause_train   = df_train.loc[fire_tr_mask, 'fire_cause'].values

# Validation: use PREDICTED fire pixels (operational simulation)
mask_pred_fire  = y_pred_ign == 1
X_pred_fire_val = X_val[mask_pred_fire]
y_cause_pred_true = df_val.loc[mask_pred_fire, 'fire_cause'].values
y_ign_pred_true   = df_val.loc[mask_pred_fire, 'target_ignition'].values

print(f"  Cause train pixels (actual fires) : {len(X_cause_train):,}")
print(f"  Predicted fire pixels (val)       : {mask_pred_fire.sum():,}")
print(f"    Actual fires in predicted set   : "
      f"{(y_ign_pred_true==1).sum():,}")
print(f"    False positives                 : "
      f"{(y_ign_pred_true==0).sum():,}")

print(f"\n  Cause distribution in training set:")
for code, label in {1:'Undetermined', 2:'Human',
                    3:'Natural', 4:'H-PB'}.items():
    n = (y_cause_train == code).sum()
    if n > 0:
        print(f"    Code {code} ({label:<15}): {n:,}  "
              f"({n/len(y_cause_train)*100:.1f}%)")

# =============================================================================
# STEP 4: Train binary cause models
# =============================================================================
print("\n" + "="*60)
print("STEP 4: Training binary cause models")
print("="*60)

# Actual fire pixels for evaluation
actual_mask   = y_ign_pred_true == 1
X_cause_val_a = X_pred_fire_val[actual_mask]
y_cause_val_a = y_cause_pred_true[actual_mask]

cause_models  = {}

for model_key, pos_label, pos_name, neg_name, color in [
    ('human',   2, 'Human',              'Other (Natural/Undetermined)', COL_HUMAN),
    ('natural', 3, 'Natural (lightning)', 'Other (Human/Undetermined)',  COL_NATURAL),
]:
    print(f"\n{'─'*55}")
    print(f"  Binary model: {pos_name} vs Other")

    # Binary labels — training on actual fire pixels
    y_tr_bin = (y_cause_train == pos_label).astype(int)
    y_vl_bin = (y_cause_val_a == pos_label).astype(int)

    n_pos_tr = y_tr_bin.sum()
    n_neg_tr = (y_tr_bin == 0).sum()
    ratio    = n_neg_tr / max(n_pos_tr, 1)

    print(f"    Train — pos: {n_pos_tr:,}  neg: {n_neg_tr:,}  "
          f"ratio: {ratio:.1f}:1")
    print(f"    Val actual fires: {len(X_cause_val_a):,}")

    t0 = time.time()
    model = RandomForestClassifier(
        n_estimators     = 300,
        max_depth        = None,
        max_features     = 'sqrt',
        min_samples_leaf = 5,
        class_weight     = 'balanced',
        oob_score        = True,
        n_jobs           = -1,
        random_state     = RANDOM_STATE,
        verbose          = 0
    )
    model.fit(X_cause_train, y_tr_bin)
    print(f"    Training time: {(time.time()-t0)/60:.1f} min  "
          f"OOB: {model.oob_score_:.4f}")

    # Evaluate on actual fires in predicted set
    y_prob_vl = model.predict_proba(X_cause_val_a)[:, 1]
    y_pred_vl = (y_prob_vl >= 0.5).astype(int)
    auc_vl    = roc_auc_score(y_vl_bin, y_prob_vl)
    ap_vl     = average_precision_score(y_vl_bin, y_prob_vl)

    print(f"    ROC-AUC: {auc_vl:.4f}  AP: {ap_vl:.4f}")
    print(classification_report(
        y_vl_bin, y_pred_vl,
        target_names=[neg_name, pos_name],
        digits=4, zero_division=0))

    save_path = f'models/model_rf_cause_binary_{model_key}.pkl'
    joblib.dump({
        'model'    : model,
        'pos_label': pos_label,
        'pos_name' : pos_name,
        'neg_name' : neg_name,
        'features' : FEATURES,
        'val_auc'  : auc_vl,
        'val_ap'   : ap_vl,
    }, save_path)
    print(f"    Saved: {save_path}")

    cause_models[model_key] = {
        'model'    : model,
        'pos_label': pos_label,
        'pos_name' : pos_name,
        'color'    : color,
        'val_auc'  : auc_vl,
    }


# =============================================================================
# STEP 4b: Detailed evaluation — binary cause models
# =============================================================================
print("\n" + "="*60)
print("STEP 4b: Detailed binary cause model evaluation")
print("="*60)

# ── Patch neg_name in case cause_models was built without it ──────────────────
NEG_NAMES = {
    'human'  : 'Other (Natural/Undetermined)',
    'natural': 'Other (Human/Undetermined)',
}

binary_eval = {}

for model_key, info in cause_models.items():
    pos_label = info['pos_label']
    pos_name  = info['pos_name']
    neg_name  = info.get('neg_name', NEG_NAMES.get(model_key, 'Other'))

    # Binary labels on actual fires in predicted set
    y_bin_true = (y_cause_val_a == pos_label).astype(int)
    y_bin_prob = info['model'].predict_proba(X_cause_val_a)[:, 1]
    y_bin_pred = (y_bin_prob >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_bin_true, y_bin_pred).ravel()
    bal_acc = balanced_accuracy_score(y_bin_true, y_bin_pred)
    prec    = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec     = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1      = (2 * prec * rec / (prec + rec)
               if (prec + rec) > 0 else 0.0)
    fpr_bin = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    spec    = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    auc_val = roc_auc_score(y_bin_true, y_bin_prob)

    binary_eval[model_key] = {
        'pos_name': pos_name,
        'neg_name': neg_name,
        'bal_acc' : bal_acc,
        'val_auc' : auc_val,
        'prec'    : prec,
        'rec'     : rec,
        'spec'    : spec,
        'f1'      : f1,
        'fpr'     : fpr_bin,
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'n_total' : len(y_bin_true),
        'n_pos'   : int(y_bin_true.sum()),
    }

    print(f"\n{'─'*55}")
    print(f"  {pos_name} vs Other — Confusion Matrix")
    print(f"  Validation: actual fires in predicted set  "
          f"(n={len(y_bin_true):,})")
    print(f"  Positive class: {pos_name}  "
          f"(n={y_bin_true.sum():,},  "
          f"{y_bin_true.mean()*100:.1f}%)")
    print(f"{'─'*55}")

    print(f"\n  Confusion matrix (threshold = 0.50):")
    print(f"  {'':25} {'Pred '+neg_name[:10]:>14} {'Pred '+pos_name[:7]:>14}")
    print(f"  {'─'*55}")
    print(f"  {'Actual '+neg_name[:10]:<25} {tn:>14,} {fp:>14,}")
    print(f"  {'Actual '+pos_name:<25} {fn:>14,} {tp:>14,}")
    print(f"  {'─'*55}")

    print(f"\n  Performance metrics:")
    print(f"    Balanced accuracy : {bal_acc:.4f}")
    print(f"    ROC-AUC           : {auc_val:.4f}")
    print(f"    Precision         : {prec:.4f}")
    print(f"    Recall            : {rec:.4f}")
    print(f"    Specificity       : {spec:.4f}")
    print(f"    F1 score          : {f1:.4f}")
    print(f"    FPR               : {fpr_bin:.4f}")
    print(f"    TP={tp:,}  FP={fp:,}  FN={fn:,}  TN={tn:,}")

# ── Side-by-side comparison table ─────────────────────────────────────────────
print(f"\n{'='*65}")
print("BINARY CAUSE MODEL COMPARISON — SIDE BY SIDE")
print(f"{'='*65}")
print(f"  {'Metric':<22} {'Human vs Other':>16} {'Natural vs Other':>18}")
print(f"  {'─'*58}")

metrics_to_show = [
    ('Balanced accuracy', 'bal_acc'),
    ('ROC-AUC',           'val_auc'),
    ('Precision',         'prec'),
    ('Recall',            'rec'),
    ('Specificity',       'spec'),
    ('F1 score',          'f1'),
    ('FPR',               'fpr'),
]

for label, key in metrics_to_show:
    h_val = binary_eval['human'][key]
    n_val = binary_eval['natural'][key]
    print(f"  {label:<22} {h_val:>16.4f} {n_val:>18.4f}")

print(f"  {'─'*58}")
for label, key in [('TP','tp'), ('FP','fp'), ('FN','fn'), ('TN','tn')]:
    h_val = binary_eval['human'][key]
    n_val = binary_eval['natural'][key]
    print(f"  {label:<22} {h_val:>16,} {n_val:>18,}")

# =============================================================================
# STEP 5b: Three-way cause prediction accuracy
# =============================================================================
print(f"\n{'='*65}")
print("STEP 5b: Three-way cause prediction accuracy")
print(f"  Uncertainty threshold : {UNCERTAIN_THRESHOLD}")
print(f"  Evaluated on          : actual fires in predicted set "
      f"(n={len(y_true_eval):,})")
print(f"{'='*65}")

n_total         = len(y_true_eval)
n_uncertain     = (y_pred_eval == 0).sum()
n_known         = (y_pred_eval != 0).sum()
correct_mask    = y_pred_eval == y_true_eval
n_correct       = correct_mask.sum()
n_correct_known = (correct_mask & (y_pred_eval != 0)).sum()

overall_acc = n_correct / n_total  if n_total > 0 else 0.0
known_acc   = n_correct_known / n_known if n_known > 0 else 0.0

# Balanced accuracy on Human + Natural only
hn_mask      = np.isin(y_true_eval, [2, 3])
y_true_hn    = y_true_eval[hn_mask]
y_pred_hn    = y_pred_eval[hn_mask]
bal_acc_3way = balanced_accuracy_score(y_true_hn, y_pred_hn)

print(f"\n  Overall accuracy (Uncertain counted as wrong):")
print(f"    {n_correct:,} / {n_total:,}  =  {overall_acc:.4f}  "
      f"({overall_acc*100:.2f}%)")

print(f"\n  Accuracy on classified predictions only (excl. Uncertain):")
print(f"    Classified : {n_known:,} / {n_total:,}  "
      f"({n_known/n_total*100:.1f}% of fires received a label)")
print(f"    Correct    : {n_correct_known:,} / {n_known:,}  "
      f"=  {known_acc:.4f}  ({known_acc*100:.2f}%)")

print(f"\n  Balanced accuracy (Human + Natural true labels only):")
print(f"    {bal_acc_3way:.4f}")

print(f"\n  Per-class breakdown:")
print(f"  {'True class':<16} {'N actual':>10} {'Pred Human':>12} "
      f"{'Pred Natural':>14} {'Pred Uncertain':>16} {'Accuracy':>10}")
print(f"  {'─'*80}")
for true_code, true_name in [(2, 'Human'), (3, 'Natural')]:
    mask_true  = y_true_eval == true_code
    n_true     = mask_true.sum()
    if n_true == 0:
        continue
    n_pred_hum = ((y_pred_eval == 2) & mask_true).sum()
    n_pred_nat = ((y_pred_eval == 3) & mask_true).sum()
    n_pred_unc = ((y_pred_eval == 0) & mask_true).sum()
    acc_c      = ((y_pred_eval == true_code) & mask_true).sum() / n_true
    print(f"  {true_name:<16} {n_true:>10,} {n_pred_hum:>12,} "
          f"{n_pred_nat:>14,} {n_pred_unc:>16,} {acc_c:>10.4f}")
print(f"  {'─'*80}")

# ── LaTeX ──────────────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("LaTeX — Binary Cause Confusion Matrices")
print(f"{'='*65}")
for model_key, e in binary_eval.items():
    pos = e['pos_name']
    print(f"\n  % {pos} vs Other  "
          f"(balanced acc={e['bal_acc']:.4f}  AUC={e['val_auc']:.4f})")
    print(f"  \\begin{{tabular}}{{lrr}}")
    print(f"  \\toprule")
    print(f"  & \\textbf{{Pred Other}} & \\textbf{{Pred {pos}}} \\\\")
    print(f"  \\midrule")
    print(f"  \\textbf{{Actual Other}}   & {e['tn']:,} & {e['fp']:,} \\\\")
    print(f"  \\textbf{{Actual {pos}}} & {e['fn']:,} & {e['tp']:,} \\\\")
    print(f"  \\bottomrule")
    print(f"  \\end{{tabular}}")

print(f"\n{'='*65}")
print("THREE-WAY CAUSE PREDICTION — FINAL SUMMARY")
print(f"{'='*65}")
print(f"  Overall accuracy      : {overall_acc:.4f}")
print(f"  Classified-only acc   : {known_acc:.4f}")
print(f"  Balanced accuracy     : {bal_acc_3way:.4f}")
print(f"  Uncertain rate        : "
      f"{n_uncertain/n_total*100:.1f}%  ({n_uncertain:,}/{n_total:,})")
# =============================================================================
# STEP 5: Three-way decision rule on predicted fire pixels
# =============================================================================
print("\n" + "="*60)
print("STEP 5: Three-way cause prediction")
print(f"  Uncertainty threshold: {UNCERTAIN_THRESHOLD}")
print("="*60)

p_human   = cause_models['human']['model'].predict_proba(
    X_pred_fire_val)[:, 1]
p_natural = cause_models['natural']['model'].predict_proba(
    X_pred_fire_val)[:, 1]

def predict_three_way(p_hum, p_nat, threshold):
    pred = np.zeros(len(p_hum), dtype=int)
    pred[(p_nat > p_hum)  & (p_nat >= threshold)] = 3  # Natural
    pred[(p_hum >= p_nat) & (p_hum >= threshold)] = 2  # Human
    return pred   # 0 = Uncertain

y_pred_cause = predict_three_way(p_human, p_natural, UNCERTAIN_THRESHOLD)

# Evaluate on actual fires only
y_true_eval = y_cause_pred_true[actual_mask]
y_pred_eval = y_pred_cause[actual_mask]

target_names_map = {0: 'Uncertain', 2: 'Human', 3: 'Natural'}
present_classes  = sorted(set(
    list(y_true_eval[np.isin(y_true_eval, [2, 3])]) +
    list(y_pred_eval)))
present_classes  = [c for c in present_classes if c in target_names_map]

print(f"\n  Prediction distribution (all predicted fire pixels):")
for code, name in [(2,'Human'), (3,'Natural'), (0,'Uncertain')]:
    n = (y_pred_cause == code).sum()
    print(f"    {name:<12}: {n:,}  ({n/len(y_pred_cause)*100:.1f}%)")

print(f"\n  Classification report (actual fires in predicted set):")
print(classification_report(
    y_true_eval, y_pred_eval,
    labels       = present_classes,
    target_names = [target_names_map[c] for c in present_classes],
    digits=4, zero_division=0))

# =============================================================================
# STEP 5b: Three-way cause prediction accuracy
# =============================================================================
print(f"\n{'='*65}")
print("STEP 5b: Three-way cause prediction accuracy")
print(f"  Uncertainty threshold : {UNCERTAIN_THRESHOLD}")
print(f"  Evaluated on          : actual fires in predicted set "
      f"(n={len(y_true_eval):,})")
print(f"{'='*65}")

n_total        = len(y_true_eval)
n_uncertain    = (y_pred_eval == 0).sum()
n_known        = (y_pred_eval != 0).sum()
correct_mask   = y_pred_eval == y_true_eval
n_correct      = correct_mask.sum()
n_correct_known = (correct_mask & (y_pred_eval != 0)).sum()

overall_acc = n_correct / n_total  if n_total > 0 else 0.0
known_acc   = n_correct_known / n_known if n_known > 0 else 0.0

# Balanced accuracy on Human + Natural only
hn_mask      = np.isin(y_true_eval, [2, 3])
y_true_hn    = y_true_eval[hn_mask]
y_pred_hn    = y_pred_eval[hn_mask]
bal_acc_3way = balanced_accuracy_score(y_true_hn, y_pred_hn)

print(f"\n  Overall accuracy (Uncertain counted as wrong):")
print(f"    {n_correct:,} / {n_total:,}  =  {overall_acc:.4f}  "
      f"({overall_acc*100:.2f}%)")

print(f"\n  Accuracy on classified predictions only (excl. Uncertain):")
print(f"    Classified : {n_known:,} / {n_total:,}  "
      f"({n_known/n_total*100:.1f}% of fires received a label)")
print(f"    Correct    : {n_correct_known:,} / {n_known:,}  "
      f"=  {known_acc:.4f}  ({known_acc*100:.2f}%)")

print(f"\n  Balanced accuracy (Human + Natural true labels only):")
print(f"    {bal_acc_3way:.4f}")

print(f"\n  Per-class breakdown:")
print(f"  {'True class':<16} {'N actual':>10} {'Pred Human':>12} "
      f"{'Pred Natural':>14} {'Pred Uncertain':>16} {'Accuracy':>10}")
print(f"  {'─'*80}")
for true_code, true_name in [(2,'Human'), (3,'Natural')]:
    mask_true   = y_true_eval == true_code
    n_true      = mask_true.sum()
    if n_true == 0:
        continue
    n_pred_hum  = ((y_pred_eval == 2) & mask_true).sum()
    n_pred_nat  = ((y_pred_eval == 3) & mask_true).sum()
    n_pred_unc  = ((y_pred_eval == 0) & mask_true).sum()
    acc_c       = ((y_pred_eval == true_code) & mask_true).sum() / n_true
    print(f"  {true_name:<16} {n_true:>10,} {n_pred_hum:>12,} "
          f"{n_pred_nat:>14,} {n_pred_unc:>16,} {acc_c:>10.4f}")
print(f"  {'─'*80}")

# ── LaTeX ──────────────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("LaTeX — Binary Cause Confusion Matrices")
print(f"{'='*65}")
for model_key, e in binary_eval.items():
    pos = e['pos_name']
    print(f"\n  % {pos} vs Other  "
          f"(balanced acc={e['bal_acc']:.4f}  AUC={e['val_auc']:.4f})")
    print(f"  \\begin{{tabular}}{{lrr}}")
    print(f"  \\toprule")
    print(f"  & \\textbf{{Pred Other}} & \\textbf{{Pred {pos}}} \\\\")
    print(f"  \\midrule")
    print(f"  \\textbf{{Actual Other}}   & {e['tn']:,} & {e['fp']:,} \\\\")
    print(f"  \\textbf{{Actual {pos}}} & {e['fn']:,} & {e['tp']:,} \\\\")
    print(f"  \\bottomrule")
    print(f"  \\end{{tabular}}")

print(f"\n{'='*65}")
print("THREE-WAY CAUSE PREDICTION — FINAL SUMMARY")
print(f"{'='*65}")
print(f"  Overall accuracy      : {overall_acc:.4f}")
print(f"  Classified-only acc   : {known_acc:.4f}")
print(f"  Balanced accuracy     : {bal_acc_3way:.4f}")
print(f"  Uncertain rate        : "
      f"{n_uncertain/n_total*100:.1f}%  ({n_uncertain:,}/{n_total:,})")
# =============================================================================
# STEP 6: Feature importance figures
# =============================================================================
print("\n" + "="*60)
print("STEP 6: Generating feature importance figures")
print("="*60)

def plot_native_importance(ax, model, title, source_note):
    imps   = model.feature_importances_
    std    = np.std([t.feature_importances_
                     for t in model.estimators_], axis=0)
    idx    = np.argsort(imps)
    feats  = [FEATURES[i]       for i in idx]
    labs   = [FEATURE_LABELS[f] for f in feats]
    colors = [feature_colour(f) for f in feats]
    vals   = imps[idx]
    stds   = std[idx]

    ax.barh(range(len(vals)), vals, xerr=stds,
            color=colors, edgecolor='white', linewidth=0.5,
            height=0.7, zorder=3,
            error_kw={'elinewidth':1.0,'capsize':2,
                      'ecolor':'#999999','capthick':1.0})
    for i, (v, s) in enumerate(zip(vals, stds)):
        ax.text(v+s+0.001, i, f'{v:.3f}',
                va='center', ha='left', fontsize=7.5, color='#333333')
    ax.set_yticks(range(len(labs)))
    ax.set_yticklabels(labs, fontsize=FONT_TICK)
    ax.set_xlabel('Importance (mean decrease impurity)', fontsize=FONT_LABEL)
    ax.set_title(title, fontsize=FONT_TITLE, fontweight='bold', pad=8)
    ax.grid(axis='x', color=COL_GRID, linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)
    ax.spines[['top','right']].set_visible(False)
    ax.set_xlim(0, (vals+stds).max() * 1.18)
    dyn_p = mpatches.Patch(facecolor=COL_DYNAMIC,
                            label='Dynamic (meteorological)')
    sta_p = mpatches.Patch(facecolor=COL_STATIC,
                            label='Static (terrain/socioeconomic)')
    ax.legend(handles=[dyn_p, sta_p], fontsize=8,
              loc='lower right', framealpha=0.9, edgecolor='#cccccc')

# ── Figure 1: Native importance — ignition + both cause models ────────────────
print("\n  Figure 1: Native importance (ignition + human + natural)...")

fig1, axes1 = plt.subplots(1, 3, figsize=(24, 7))
fig1.subplots_adjust(wspace=0.38)

plot_native_importance(
    axes1[0], model_rf_ign,
    '(a) Ignition Model\nNative Feature Importance', SOURCE_IGN)

for ax, key, letter in [(axes1[1],'human','b'), (axes1[2],'natural','c')]:
    info  = cause_models[key]
    title = (f'({letter}) {info["pos_name"]} vs Other\n'
             f'Cause Model — Native Importance')
    plot_native_importance(ax, info['model'], title, SOURCE_CAUSE)

fig1.suptitle(
    'Random Forest — Native Feature Importance (Mean Decrease in Impurity)\n'
    'Ignition Model + Binary Cause Models',
    fontsize=13, fontweight='bold', y=1.01)
fig1.text(0.5, -0.01, SOURCE_CAUSE,
          ha='center', fontsize=7.5, color='#888888', style='italic')
fig1.savefig('figures/fig_rf_native_importance_all.png',
             dpi=300, bbox_inches='tight', facecolor='white')
print("    Saved: figures/fig_rf_native_importance_all.png")
plt.close(fig1)

# ── Figure 2: Native importance — ignition only (standalone) ──────────────────
fig2, ax2 = plt.subplots(figsize=(10, 8))
plot_native_importance(
    ax2, model_rf_ign,
    'Ignition Model — Native Feature Importance\n'
    'Random Forest  |  Mean Decrease in Impurity', SOURCE_IGN)
fig2.text(0.5, -0.01, SOURCE_IGN,
          ha='center', fontsize=7.5, color='#888888', style='italic')
plt.tight_layout()
fig2.savefig('figures/fig_rf_feature_importance.png',
             dpi=300, bbox_inches='tight', facecolor='white')
print("    Saved: figures/fig_rf_feature_importance.png")
plt.close(fig2)

# =============================================================================
# STEP 7: Permutation importance
# =============================================================================
print("\n" + "="*60)
print("STEP 7: Permutation importance")
print("="*60)

# ── Ignition permutation importance ───────────────────────────────────────────
print("\n  Computing ignition permutation importance...")
PERM_NOFIRE_SAMPLE = 50_000

fire_val_pi   = df_val[df_val['target_ignition'] == 1]
nofire_val_pi = df_val[df_val['target_ignition'] == 0].sample(
    n=min(PERM_NOFIRE_SAMPLE, (df_val['target_ignition']==0).sum()),
    random_state=RANDOM_STATE)
perm_df = pd.concat([fire_val_pi, nofire_val_pi], ignore_index=True)
X_perm_ign = perm_df[FEATURES].values
y_perm_ign = perm_df['target_ignition'].values

print(f"  PI sample: {len(fire_val_pi):,} fire + "
      f"{len(nofire_val_pi):,} no-fire")

t0 = time.time()
pi_ign = permutation_importance(
    model_rf_ign, X_perm_ign, y_perm_ign,
    n_repeats=10, scoring='roc_auc',
    random_state=RANDOM_STATE, n_jobs=-1)
print(f"  Computed in {(time.time()-t0)/60:.1f} minutes")

# ── Cause permutation importance (on actual fires in predicted set) ────────────
print("\n  Computing cause model permutation importance...")
pi_cause = {}
for key, info in cause_models.items():
    print(f"    {info['pos_name']} vs Other...")
    y_bin = (y_cause_val_a == info['pos_label']).astype(int)
    t0 = time.time()
    pi = permutation_importance(
        info['model'], X_cause_val_a, y_bin,
        n_repeats=10, scoring='roc_auc',
        random_state=RANDOM_STATE, n_jobs=-1)
    print(f"    Done in {(time.time()-t0)/60:.1f} minutes")
    pi_cause[key] = pi

# ── Plot helper ────────────────────────────────────────────────────────────────
def plot_pi(ax, pi_result, title):
    means  = pi_result.importances_mean
    stds   = pi_result.importances_std
    idx    = np.argsort(means)
    feats  = [FEATURES[i]       for i in idx]
    labs   = [FEATURE_LABELS[f] for f in feats]
    colors = [feature_colour(f) for f in feats]

    ax.barh(range(len(idx)), means[idx], xerr=stds[idx],
            color=colors, edgecolor='white', linewidth=0.5,
            height=0.7, zorder=3,
            error_kw={'elinewidth':1.2,'capsize':3,
                      'ecolor':'#666666','capthick':1.2})
    ax.axvline(0, color='#333333', linewidth=1.0,
               linestyle='--', zorder=4)
    for i, (m, s) in enumerate(zip(means[idx], stds[idx])):
        if m > 0:
            ax.text(m+s+0.0003, i, f'{m:.4f}',
                    va='center', ha='left',
                    fontsize=7.5, color='#333333')
    ax.set_yticks(range(len(labs)))
    ax.set_yticklabels(labs, fontsize=FONT_TICK)
    ax.set_xlabel('Decrease in ROC-AUC when permuted\n'
                  '(mean ± std, 10 repeats)', fontsize=FONT_LABEL)
    ax.set_title(title, fontsize=FONT_TITLE, fontweight='bold', pad=8)
    ax.grid(axis='x', color=COL_GRID, linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)
    ax.spines[['top','right']].set_visible(False)
    dyn_p = mpatches.Patch(facecolor=COL_DYNAMIC,
                            label='Dynamic (meteorological)')
    sta_p = mpatches.Patch(facecolor=COL_STATIC,
                            label='Static (terrain/socioeconomic)')
    ax.legend(handles=[dyn_p, sta_p], fontsize=8,
              loc='lower right', framealpha=0.9, edgecolor='#cccccc')

# ── Figure 3: PI — all three models ───────────────────────────────────────────
print("\n  Figure 3: Permutation importance (all 3 models)...")

fig3, axes3 = plt.subplots(1, 3, figsize=(24, 7))
fig3.subplots_adjust(wspace=0.38)

plot_pi(axes3[0], pi_ign,
        '(a) Ignition Model\nPermutation Importance')
for ax, key, letter in [(axes3[1],'human','b'), (axes3[2],'natural','c')]:
    info = cause_models[key]
    plot_pi(ax, pi_cause[key],
            f'({letter}) {info["pos_name"]} vs Other\n'
            f'Permutation Importance')

fig3.suptitle(
    'Random Forest — Permutation Feature Importance (ROC-AUC)\n'
    'Ignition Model + Binary Cause Models  |  Validation: 2021–2022',
    fontsize=13, fontweight='bold', y=1.01)
fig3.text(0.5, -0.01, SOURCE_CAUSE,
          ha='center', fontsize=7.5, color='#888888', style='italic')
fig3.savefig('figures/fig_rf_permutation_importance_all.png',
             dpi=300, bbox_inches='tight', facecolor='white')
print("    Saved: figures/fig_rf_permutation_importance_all.png")
plt.close(fig3)

# ── Figure 4: PI — ignition only (standalone for thesis) ──────────────────────
fig4, ax4 = plt.subplots(figsize=(10, 8))
plot_pi(ax4, pi_ign,
        'Permutation Feature Importance — RF Ignition Model\n'
        'Validation set: 2021–2022  |  Metric: ROC-AUC')
fig4.text(0.5, -0.01, SOURCE_IGN,
          ha='center', fontsize=7.5, color='#888888', style='italic')
plt.tight_layout()
fig4.savefig('figures/fig_rf_permutation_importance.png',
             dpi=300, bbox_inches='tight', facecolor='white')
print("    Saved: figures/fig_rf_permutation_importance.png")
plt.close(fig4)

# ── Figure 5: Cause models — native + PI side by side (2×2) ───────────────────
print("\n  Figure 5: Cause model importance comparison (2x2)...")

fig5, axes5 = plt.subplots(2, 2, figsize=(20, 14))
fig5.subplots_adjust(wspace=0.35, hspace=0.40)

for col, key, label in [(0,'human','Human vs Other'),
                         (1,'natural','Natural vs Other')]:
    plot_native_importance(
        axes5[0][col], cause_models[key]['model'],
        f'{label}\nNative Importance', SOURCE_CAUSE)
    plot_pi(
        axes5[1][col], pi_cause[key],
        f'{label}\nPermutation Importance')

fig5.suptitle(
    'Binary Cause Models — Feature Importance Comparison\n'
    'Random Forest  |  Native (MDI) vs Permutation (ROC-AUC)',
    fontsize=13, fontweight='bold', y=1.01)
fig5.text(0.5, -0.01, SOURCE_CAUSE,
          ha='center', fontsize=7.5, color='#888888', style='italic')
fig5.savefig('figures/fig_rf_cause_importance_comparison.png',
             dpi=300, bbox_inches='tight', facecolor='white')
print("    Saved: figures/fig_rf_cause_importance_comparison.png")
plt.close(fig5)

# =============================================================================
# STEP 8: Final summary
# =============================================================================
print(f"\n{'='*65}")
print("FINAL SUMMARY")
print(f"{'='*65}")
print(f"\n  IGNITION MODEL")
print(f"    ROC-AUC     : {roc_auc_score(y_ign_val, y_prob_ign):.4f}")
print(f"    Threshold   : {IGNITION_THRESHOLD}")
print(f"    Recall      : {tp/(tp+fn):.4f}  ({tp}/{tp+fn} fires detected)")
print(f"    Precision   : {tp/(tp+fp) if (tp+fp)>0 else 0:.4f}")
print(f"    FPR         : {fp/(fp+tn):.4f}")
print(f"    OOB score   : {model_rf_ign.oob_score_:.4f}")

print(f"\n  CAUSE MODELS (binary RF, three-way decision)")
print(f"    Uncertainty threshold : {UNCERTAIN_THRESHOLD}")
for key, info in cause_models.items():
    print(f"    {info['pos_name']:<22}: "
          f"ROC-AUC = {info['val_auc']:.4f}")

print(f"\n  Prediction distribution (actual fires in predicted set):")
for code, name in [(2,'Human'), (3,'Natural'), (0,'Uncertain')]:
    n = (y_pred_eval == code).sum()
    print(f"    {name:<12}: {n:,}  ({n/len(y_pred_eval)*100:.1f}%)")

print(f"\n  Models saved:")
print(f"    models/model_rf_ignition_2010_2022_ratio20.pkl")
print(f"    models/model_rf_cause_binary_human.pkl")
print(f"    models/model_rf_cause_binary_natural.pkl")

print(f"\n  Figures saved:")
for fig_name in [
    'fig_rf_feature_importance.png         — ignition MDI (standalone)',
    'fig_rf_permutation_importance.png     — ignition PI (standalone)',
    'fig_rf_native_importance_all.png      — all 3 models MDI',
    'fig_rf_permutation_importance_all.png — all 3 models PI',
    'fig_rf_cause_importance_comparison.png — cause MDI vs PI (2x2)',
]:
    print(f"    figures/{fig_name}")

print(f"\nLaTeX:")
print(f"  \\includegraphics[width=\\textwidth]"
      f"{{figures/fig_rf_native_importance_all}}")
print(f"  \\includegraphics[width=\\textwidth]"
      f"{{figures/fig_rf_permutation_importance_all}}")
print(f"  \\includegraphics[width=\\textwidth]"
      f"{{figures/fig_rf_cause_importance_comparison}}")